In [2]:
import torch
import torch.nn as nn
import numpy as np

In [104]:
D = 16

padding = 1
kernel_size = 3
stride = 3
layers = 4 #not changing the model yet!!!!

In [105]:
def compute_compressed_size_decoder(L_out, layers, kernel_size, stride, padding, output_padding=0):

    L = L_out
    list_L = [L]
    # print('\n', L)

    for _ in range(layers):
        
        # L = (L + 2*padding - kernel_size + output_padding) // stride + 1 
        # print(L)

        L = (L + 2*padding - kernel_size + output_padding) / stride + 1 
        print(L)
        L = int( np.ceil( L ) )
        print("to ", L)
        print('leading to out shape ', (L-1)*stride - 2*padding + kernel_size)

        list_L.append(L)
        print(list_L[::-1])
    
    for l in list_L[::-1]:
        print((l-1)*stride - 2*padding + kernel_size)

    return list_L[::-1]

In [102]:
class Conv1DDecoder(nn.Module):

    def __init__(self, output_shape, D, layers=2,
                 kernel_size=3, stride=2, padding=1):
        super().__init__()

        self.D = D
        self.layers = layers
        self.n_var = output_shape[0]
        self.ts_var = output_shape[1]

        self.list_ts_comp = compute_compressed_size_decoder(
            self.ts_var, layers, kernel_size, stride, padding
        )

        # final encoder channels
        final_channels = D * (2 ** (layers - 1))

        self.fc = nn.Linear(D, final_channels * self.list_ts_comp[0])

        modules = []
        in_channels = final_channels

        for i in range(layers):
            out_channels = in_channels // 2 if i < layers-1 else self.n_var

            modules.append(
                nn.ConvTranspose1d(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    stride=stride,
                    padding=padding,
                    # output_padding=1
                )
            )

            if i < layers-1:
                modules.append(nn.ReLU())
                modules.append(nn.BatchNorm1d(out_channels))

            in_channels = out_channels

        self.transposecnn = nn.Sequential(*modules)

    def forward(self, x):
        
        print('In Decoder: ', x.shape)
        x = self.fc(x)
        print('After FC: ', x.shape)
        
        x = x.view(-1,
                   self.D * (2 ** (self.layers - 1)),
                   self.list_ts_comp[0])
        print('After reshape: ', x.shape)

        conv_id = 0  # counts ConvTranspose1d layers only

        for i, layer in enumerate(self.transposecnn):
            x = layer(x)
            if isinstance(layer, nn.ConvTranspose1d):
                conv_id += 1
                target_len = self.list_ts_comp[conv_id]  # next expected length
                x = x[:, :, :target_len]
                print(f"After ConvTranspose crop {conv_id}: {x.shape}")

        # crop/pad to original length
        # x = x[:, :, :self.ts_var]
        print('Out shape : ', x.shape)

        return x

In [106]:
B = 128

var_shape = (1, 5000)
model = Conv1DDecoder(var_shape, D, layers, kernel_size, stride, padding)

input = torch.randn((B,D)) 
model(input)


from torchinfo import summary
shape_input = (B,D)
summary(model, input_size=(shape_input))

1667.3333333333333
to  1668
leading to out shape  5002
[1668, 5000]
556.6666666666666
to  557
leading to out shape  1669
[557, 1668, 5000]
186.33333333333334
to  187
leading to out shape  559
[187, 557, 1668, 5000]
63.0
to  63
leading to out shape  187
[63, 187, 557, 1668, 5000]
187
559
1669
5002
14998
In Decoder:  torch.Size([128, 16])
After FC:  torch.Size([128, 8064])
After reshape:  torch.Size([128, 128, 63])
After ConvTranspose crop 1: torch.Size([128, 64, 187])
After ConvTranspose crop 2: torch.Size([128, 32, 557])
After ConvTranspose crop 3: torch.Size([128, 16, 1668])
After ConvTranspose crop 4: torch.Size([128, 1, 5000])
Out shape :  torch.Size([128, 1, 5000])
In Decoder:  torch.Size([128, 16])
After FC:  torch.Size([128, 8064])
After reshape:  torch.Size([128, 128, 63])
After ConvTranspose crop 1: torch.Size([128, 64, 187])
After ConvTranspose crop 2: torch.Size([128, 32, 557])
After ConvTranspose crop 3: torch.Size([128, 16, 1668])
After ConvTranspose crop 4: torch.Size([128

Layer (type:depth-idx)                   Output Shape              Param #
Conv1DDecoder                            [128, 1, 5000]            --
├─Linear: 1-1                            [128, 8064]               137,088
├─Sequential: 1-2                        --                        --
│    └─ConvTranspose1d: 2-1              [128, 64, 187]            24,640
│    └─ReLU: 2-2                         [128, 64, 187]            --
│    └─BatchNorm1d: 2-3                  [128, 64, 187]            128
│    └─ConvTranspose1d: 2-4              [128, 32, 559]            6,176
│    └─ReLU: 2-5                         [128, 32, 557]            --
│    └─BatchNorm1d: 2-6                  [128, 32, 557]            64
│    └─ConvTranspose1d: 2-7              [128, 16, 1669]           1,552
│    └─ReLU: 2-8                         [128, 16, 1668]           --
│    └─BatchNorm1d: 2-9                  [128, 16, 1668]           32
│    └─ConvTranspose1d: 2-10             [128, 1, 5002]            49